In [ ]:
!pip install pydrive --upgrade


from google.colab import auth
auth.authenticate_user()
import gspread
from google.auth import default
import pandas as pd
import openpyxl
import requests
import numpy as np
from io import BytesIO
import datetime as dt
creds, _ = default()
import regex as re


gc = gspread.authorize(creds)

# Import PyDrive and associated libraries.
# This only needs to be done once per notebook.
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

# Authenticate and create the PyDrive client.
# This only needs to be done once per notebook.
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 987.4/987.4 kB 7.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pydrive: filename=PyDrive-1.3.1-py3-none-any.whl size=27433 sha256=614c3c3218d7192adec9ea46628715895d86581dc0ce768c8abca882eead75f9
  Stored in directory: /root/.cache/pip/wheels/6c/10/da/a5b513f5b3916fc391c20ee7b4633e5cf3396d570cdd74970f
Successfully built pydrive


In [ ]:
headers = ["Work Type","Practice", "Claim ID", "Worked By", "Work Date","ClaimFlag","Work Status"]

listOfFrames_1 = []

conn = gspread.authorize(creds)
sheets = [
          '1msfbrNKRF8iDvLxwS9TgvJKVibZXt8XLqcFW_qnOylc',       #Claims Billing Tracker - Nov 2025
          '1RPOinQZG4QQjsxhr2Zsi-kwmQxR7dAMlYpvqDYJGV6E',       #Claims Billing Tracker - Dec 2025
          '172ZNGOfYtDtjEqMLPKaaPgSqFufO4vfuzItLelWjIl4'   ]    #Claims Billing Tracker - Jan 2026




for sheet in sheets:  # Added
    worksheet_list = conn.open_by_key(sheet).worksheet("Rejections/Resub")
    rows = worksheet_list.get_all_values()
    data = zip(*(e for e in zip(*rows) if e[0].strip() in headers))
    df = pd.DataFrame(data, columns=headers)
    df.rename(columns = df.iloc[0].apply(lambda x: x.strip()), inplace = True)
    df.drop(df.index[0], inplace = True)
    listOfFrames_1.append(df)
    print(sheet)

1msfbrNKRF8iDvLxwS9TgvJKVibZXt8XLqcFW_qnOylc
1RPOinQZG4QQjsxhr2Zsi-kwmQxR7dAMlYpvqDYJGV6E
172ZNGOfYtDtjEqMLPKaaPgSqFufO4vfuzItLelWjIl4


In [ ]:
combinedDF1 = pd.concat([df.reset_index(drop=True) for df in listOfFrames_1],
                        axis=0, ignore_index=True)

In [ ]:
combinedDF1['Work Status'] = combinedDF1['Work Status'].apply(str)

In [ ]:
combinedDF1 = combinedDF1.drop_duplicates()
combinedDF1 = combinedDF1.drop_duplicates(subset=['Practice', 'Claim ID', 'Work Date','Worked By'])
combinedDF1.shape

(9721, 7)

In [ ]:
# Assuming 'claimflag' exists in your DataFrame and you want to filter by it for resubmissions.

# First, separate the data into two subsets based on 'Work Type'
rejection_df = combinedDF1[combinedDF1['Work Type'] == 'Rejections']
resubmission_df = combinedDF1[combinedDF1['Work Type'] == 'Resubmissions']

In [ ]:
# Apply the same filter (non-null and non-empty Work Status) to rejection data
rejection_df = rejection_df.loc[~rejection_df['ClaimFlag'].isnull()]
rejection_df = rejection_df[rejection_df['ClaimFlag'].notnull() & (rejection_df['ClaimFlag'] != '')]

In [ ]:
# For resubmission data, apply filter based on 'claimflag' column
# (Assuming 'claimflag' exists in your DataFrame, replace this logic with the condition you want to apply)
resubmission_df = resubmission_df.loc[~resubmission_df['Work Status'].isnull()]
resubmission_df = resubmission_df[resubmission_df['Work Status'].notnull() & (resubmission_df['Work Status'] != '')]

In [ ]:
# Create a regular expression pattern that matches any of the unwanted statuses
pattern = r"(Duplicate|Already done|Not Billed|None|Already worked|On Hold)"

In [ ]:
combinedDF1 = resubmission_df.loc[~resubmission_df['Work Status'].str.contains(pattern, flags=re.IGNORECASE, regex=True)]
combinedDF1.shape

/tmp/ipython-input-2091568404.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  combinedDF1 = resubmission_df.loc[~resubmission_df['Work Status'].str.contains(pattern, flags=re.IGNORECASE, regex=True)]


(3487, 7)

In [ ]:
# Create a regular expression pattern that matches any of the unwanted statuses
Exclude = r"(Already worked|Claim in Process)"

In [ ]:
combinedDF2 = rejection_df.loc[~rejection_df['ClaimFlag'].str.contains(Exclude, flags=re.IGNORECASE, regex=True)]
combinedDF2.shape

/tmp/ipython-input-1350345932.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  combinedDF2 = rejection_df.loc[~rejection_df['ClaimFlag'].str.contains(Exclude, flags=re.IGNORECASE, regex=True)]


(4690, 7)

In [ ]:
combinedDF1 = combinedDF1.loc[~combinedDF1['Work Date'].isnull()]
combinedDF1 = combinedDF1[combinedDF1['Work Date'].notnull() & (combinedDF1['Work Date'] != '')]
combinedDF1.shape

(3476, 7)

In [ ]:
combinedDF2 = combinedDF2.loc[~combinedDF2['Work Date'].isnull()]
combinedDF2 = combinedDF2[combinedDF2['Work Date'].notnull() & (combinedDF2['Work Date'] != '')]
combinedDF2.shape

(4689, 7)

In [ ]:
combinedDF1 = combinedDF1.loc[~combinedDF1['Worked By'].isnull()]
combinedDF1 = combinedDF1[combinedDF1['Worked By'].notnull() & (combinedDF1['Worked By'] != '')]
combinedDF1.shape

(3476, 7)

In [ ]:
combinedDF2 = combinedDF2.loc[~combinedDF2['Worked By'].isnull()]
combinedDF2 = combinedDF2[combinedDF2['Worked By'].notnull() & (combinedDF2['Worked By'] != '')]
combinedDF2.shape

(4689, 7)

In [ ]:
combinedDF1 = combinedDF1.sort_values('Work Date')
combinedDF1 = combinedDF1.fillna("")
combinedDF1.shape

(3476, 7)

In [ ]:
combinedDF2 = combinedDF2.sort_values('Work Date')
combinedDF2 = combinedDF2.fillna("")
combinedDF2.shape

(4689, 7)

In [ ]:
combinedDF1['Work Date'] = pd.to_datetime(combinedDF1['Work Date'],errors = 'coerce')
combinedDF1.shape

(3476, 7)

In [ ]:
combinedDF2['Work Date'] = pd.to_datetime(combinedDF2['Work Date'],errors = 'coerce')
combinedDF2.shape

(4689, 7)

In [ ]:
combinedDF1['Claim ID'] = pd.to_numeric(combinedDF1['Claim ID'], errors='coerce').astype('Int64')

In [ ]:
combinedDF2['Claim ID'] = pd.to_numeric(combinedDF2['Claim ID'], errors='coerce').astype('Int64')

In [ ]:
# Pivot the DataFrame to get Work Type as columns, aggregated by Worked Date and Worked By
pivot_df_Resubmissions = combinedDF1.pivot_table(
    index=['Work Date', 'Worked By'],    # Set both 'Worked Date' and 'Worked By' as indices
    columns='Work Type',                   # Set 'Work Type' as columns
    aggfunc='size',                        # Count the occurrences of each Work Type
    fill_value=0                           # Fill missing values with 0
)

# Remove any blank column caused by multi-indexing in the 'columns' level
pivot_df_Resubmissions.columns.name = None  # This removes the 'Work Type' name in the column headers

# Reset index if you prefer a flat DataFrame
pivot_df_Resubmissions = pivot_df_Resubmissions.reset_index()

# View the reshaped DataFrame
print(pivot_df_Resubmissions)

     Work Date      Worked By  Resubmissions
0   2025-11-03    Braison L H              2
1   2025-11-03     Reshma B L             16
2   2025-11-03     Sreejith S             18
3   2025-11-04     Anandu V S             33
4   2025-11-04    Braison L H             11
..         ...            ...            ...
228 2026-01-06    Braison L H              3
229 2026-01-06    Govindh V R              5
230 2026-01-06       Sarath S              4
231 2026-01-06  Sayed Ahammed             41
232 2026-01-06        Shiva G              1

[233 rows x 3 columns]


In [ ]:
# Pivot the DataFrame to get Work Type as columns, aggregated by Worked Date and Worked By
pivot_df_rejections = combinedDF2.pivot_table(
    index=['Work Date', 'Worked By'],    # Set both 'Worked Date' and 'Worked By' as indices
    columns='Work Type',                   # Set 'Work Type' as columns
    aggfunc='size',                        # Count the occurrences of each Work Type
    fill_value=0                           # Fill missing values with 0
)

# Remove any blank column caused by multi-indexing in the 'columns' level
pivot_df_rejections.columns.name = None  # This removes the 'Work Type' name in the column headers

# Reset index if you prefer a flat DataFrame
pivot_df_rejections = pivot_df_rejections.reset_index()

# View the reshaped DataFrame
print(pivot_df_rejections)

     Work Date      Worked By  Rejections
0   2025-11-03      Hybin J S          63
1   2025-11-03      Rithika R           4
2   2025-11-03        Shiva G          34
3   2025-11-04      Hybin J S          65
4   2025-11-04      Rithika R          23
..         ...            ...         ...
145 2026-01-05        Shiva G           1
146 2026-01-06    Braison L H           4
147 2026-01-06      Rithika R          47
148 2026-01-06  Sayed Ahammed           6
149 2026-01-06        Shiva G           6

[150 rows x 3 columns]


In [ ]:
# Step 1: Merge the pivot tables on 'Work Date' and 'Worked By'
combined_pivot_df = pd.merge(
    pivot_df_Resubmissions[['Work Date', 'Worked By', 'Resubmissions']],  # Select only relevant columns from Resubmissions pivot
    pivot_df_rejections[['Work Date', 'Worked By', 'Rejections']],       # Select only relevant columns from Rejections pivot
    on=['Work Date', 'Worked By'],                                       # Merge on 'Work Date' and 'Worked By'
    how='outer'                                                          # Outer join to keep all rows, filling missing with NaN
)

# Step 2: Fill NaN values with 0 (in case there were no rejections or resubmissions for a specific Work Date and Worked By)
combined_pivot_df = combined_pivot_df.fillna(0)

# Step 3: Optionally, convert to integer if needed (since the counts are integers)
combined_pivot_df['Rejections'] = combined_pivot_df['Rejections'].astype(int)
combined_pivot_df['Resubmissions'] = combined_pivot_df['Resubmissions'].astype(int)

# Print the combined DataFrame
print(combined_pivot_df)

     Work Date      Worked By  Resubmissions  Rejections
0   2025-11-03    Braison L H              2           0
1   2025-11-03      Hybin J S              0          63
2   2025-11-03     Reshma B L             16           0
3   2025-11-03      Rithika R              0           4
4   2025-11-03        Shiva G              0          34
..         ...            ...            ...         ...
329 2026-01-06    Govindh V R              5           0
330 2026-01-06      Rithika R              0          47
331 2026-01-06       Sarath S              4           0
332 2026-01-06  Sayed Ahammed             41           6
333 2026-01-06        Shiva G              1           6

[334 rows x 4 columns]


In [ ]:
combined_pivot_df = combined_pivot_df.sort_values('Work Date')
combined_pivot_df['Work Date'] = combined_pivot_df['Work Date'].dt.date
combined_pivot_df['Work Date'] = combined_pivot_df['Work Date'].apply(str)
combined_pivot_df["Worked By"] = combined_pivot_df["Worked By"].str.strip()

In [ ]:
combined_pivot_df = combined_pivot_df.fillna(0)

In [ ]:
combinedDFPivot = [combined_pivot_df.columns.to_list()] + combined_pivot_df.to_numpy().tolist()

In [ ]:
wsMaster = gc.open_by_key("1ffYld-zvM4kMWB3c47GVqsmgoU8S2jPrwUG4U6fWXY8").worksheet("Rejection/Resubmission")
wsMaster.clear()
wsMaster.update("A1",combinedDFPivot,value_input_option="USER_ENTERED")

/tmp/ipython-input-3662677925.py:3: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  wsMaster.update("A1",combinedDFPivot,value_input_option="USER_ENTERED")


{'spreadsheetId': '1ffYld-zvM4kMWB3c47GVqsmgoU8S2jPrwUG4U6fWXY8',
 'updatedRange': "'Rejection/Resubmission'!A1:D335",
 'updatedRows': 335,
 'updatedColumns': 4,
 'updatedCells': 1340}